In [1]:
import jpype
import os
# jvm 강제 종료
if jpype.isJVMStarted():
  jpype.shutdownJVM()
  
# 환경 변수에 JAVA_HOME이라는 변수들 등록
os.environ['JAVA_HOME'] = r'C:\Program Files\Java\jdk-21.0.10'

# 일반 문자열에서 \는 기능이 있음. \문자를 하려면 \\해야함
from konlpy.tag import Okt
okt = Okt()

In [2]:
# 텍스트 마이닝
# 문장을 분석해서 유사한 문장을 선택하도록 학습
# 텍스트 전처리 : 불필요한 공백, 특수문자, 의미없는 단어를 제거
# 토큰화 : 문장을 단어 단위로 쪼갬
# 백터화 : 단어를 컴퓨터가 계산할 수 있는 숫자(벡터)로 변환

In [3]:
# 단어가방(Bag of Words : Bow)
# 문장을 단어와 출현 회수로 이루어진 고유한 맵으로 변환하여 백터화 하는 과정

In [4]:
# 단어 가방 예제

In [5]:
# 문장들 생성
corpus = ['강사님은 파이썬을 좋아합니다', '강사님은 자바도 좋아합니다',
          '파이썬은 어렵습니다', '학생들도 파이썬을 좋아합니다']

In [6]:
# BoW 변환
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()
bow_vector = vectorizer.fit_transform(corpus)


In [7]:
vectorizer.vocabulary_ # 추출한 단어 목록들

{'강사님은': 0, '파이썬을': 5, '좋아합니다': 3, '자바도': 2, '파이썬은': 4, '어렵습니다': 1, '학생들도': 6}

In [8]:
bow_vector.toarray()
# 강사님은 어렵습니다 자바도 좋아합니다 파이썬은 파이썬을 학생들도


array([[1, 0, 0, 1, 0, 1, 0],
       [1, 0, 1, 1, 0, 0, 0],
       [0, 1, 0, 0, 1, 0, 0],
       [0, 0, 0, 1, 0, 1, 1]])

In [9]:
# 형태소 분석
# 한국어는 단어 뒤에 조사 붙음 => 같은 단어여도 조사에 따라 다른 단어로 판별될 수 있음
# 형태소 분석기
# 문장을 분석하여 의미가 있는 최소 단위로 쪼개고, 각 단어가 명사, 동사, 조사인지 등의 태그를 붙임

In [10]:
# KoNLPy 라이브러리 설치 완료
# !pip install konlpy

In [11]:
from konlpy.tag import Okt

In [12]:
# okt = Okt()
# text = "강사님은 파이썬을 좋아합니다"
# # 형태소 단위로 쪼갬
# okt.morphs()

In [13]:
# TF-IDF(Term Frequency-Inverse Document Frequency)
# 반복되서 나오는 단어는 중요하지만, 모든 문서에서 나오는 반복된 단어는 중요도가 낮음
# TF : 단어가 해당 문서에 얼마나 등장하는가? => 많을 수록 좋음
# IDF  : 해당 단어가 얼마나 많은 문서에 공통으로 등장하는가? => 많을 수록 감정

In [14]:
# TD-IDF 예제

In [15]:
# 데이터 생성
corpus = ['강사님은 파이썬을 좋아합니다', '강사님은 자바도 좋아합니다',
          '파이썬은 어렵습니다', '학생들도 파이썬을 좋아합니다']

In [16]:
okt = Okt()

# 주어진 문자열들을 명사로만된 문자열들로 바꾸는 과정
# 형태소 분석 및 명사 추출
clean_corpus = []
for text in corpus:
  # 명사들만 추출
  nouns = okt.nouns(text)
  # 명사들을 하나의 문자열로 변환
  clean_text = " ".join(nouns)
  # 결과 리스트에 추가
  clean_corpus.append(clean_text)
  
print(clean_corpus)


['강사 파이썬', '강사 자바', '파이썬', '학생 파이썬']


In [17]:
# 명사, 동사, 형용사를 골라내는 작업
clean_corpus2 = []

for text in corpus:
	# stem=True : 동사를 동사 원형으로 바꿈
	result = okt.pos(text, stem=True)
	words = []


	for word, pos in result:
		# print(f'{word}({pos})')
		
		# 단어의 품사가 명사, 동사, 형용사이면 추출하여 리스트에 추가
		if pos in ['Noun', 'Verb', 'Adjective']:
			words.append(word)
			
	# 리스트에 있는 단어들을 문자열로 변환 => 명사, 동사, 형용사만 있는 문자로 변환
	new_text = " ".join(words)
	clean_corpus2.append(new_text)
	# print(new_text)
 

In [18]:
# 유사한지를 확인할 새로운 문장을 추가
query = '학생들은 c언어를 좋아합니다'

result = okt.pos(query, stem=True)
words = []

for word, pos in result:
	# print(f'{word}({pos})')
	
	# 단어의 품사가 명사, 동사, 형용사이면 추출하여 리스트에 추가
	if pos in ['Noun', 'Verb', 'Adjective']:
		words.append(word)
		
# 리스트에 있는 단어들을 문자열로 변환 => 명사, 동사, 형용사만 있는 문자로 변환
new_query = " ".join(words)
new_query

'학생 언어 좋아하다'

In [19]:
# TF-IDF를 적용하여 벡터화
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(clean_corpus2 + [new_query])
# query_matrix = vectorizer.fit_transform([new_query])
# tfidf_matrix.toarray()

In [ ]:
# 코사인 유사도를 이용하여 가장 유사한 문장을 찾음
from sklearn.metrics.pairwise import cosine_similarity
similarity = cosine_similarity(tfidf_matrix[-1],tfidf_matrix[:-1])

import numpy as np

max_idx = np.argmax(similarity[0])
print(f'"{query}"와 가장 유사한 문장 : "{corpus[max_idx]}')


array([0.1900643 , 0.16125455, 0.        , 0.5798436 ])